# 🧠 Módulo 04: La Revolución del Transformer y la Atención
## Capítulo 3: Construcción Integral de un LLM Moderno: De Tokenizador BPE a NanoLLaMA con KV-Cache

> *"Todo gran modelo de lenguaje moderno (GPT-4, LLaMA 3, Claude 3, DeepSeek) comparte un mismo esqueleto arquitectónico: un tokenizador de subpalabras (BPE), una pila de bloques decodificadores autorregresivos con Pre-RMSNorm y RoPE, y un motor de inferencia acelerado mediante KV-Cache. En este capítulo ensamblaremos todas las piezas desde primeros principios para dar vida a un LLM completo y funcional."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/04_transformers/03_nanogpt_from_scratch.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías necesarias y fijamos semillas para asegurar reproducibilidad determinista.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict, Optional
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para construir NanoLLaMA con KV-Cache from scratch")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### De Codificador-Decodificador (2017) a Decodificadores Puros (GPT, 2018 - 2020)
El Transformer original de Vaswani et al. (2017) fue concebido para traducción automática con dos mitades:
* Un **Codificador** que leía la frase de entrada completa con atención bidireccional.
* Un **Decodificador** que generaba la traducción paso a paso con atención causal y cruzada.

En 2018, **Alec Radford y el equipo de OpenAI** tuvieron una revelación fundamental con **GPT (Generative Pre-trained Transformer)**:
> *Si prescindimos del codificador y entrenamos únicamente un decodificador autorregresivo profundo con la tarea auto-supervisada de **predecir el siguiente token** sobre millones de textos de Internet, el modelo aprende implícitamente gramática, lógica, conocimiento del mundo y capacidad de razonamiento.* Nació la era de los LLMs modernos.

### La Receta Arquitectónica Moderna (LLaMA, Mistral, Gemma)
Entre 2020 y 2024, la arquitectura original de GPT-3 evolucionó hacia un estándar refinado adoptado por toda la industria de código abierto:
1. **Pre-RMSNorm (Zhang & Sennrich, 2019):** Reemplazo de LayerNorm por RMSNorm en la entrada de cada bloque, manteniendo la autopista residual $x + \mathcal{F}(x)$ pura y estable.
2. **Rotary Position Embeddings (RoPE, Su et al., 2021):** Rotación de consultas y claves en el plano complejo en lugar de embeddings posicionales absolutos sumados.
3. **SwiGLU Activation (Shazeer, 2020):** Multiplicación por compuerta con activación SiLU/Swish en el MLP, superando a ReLU en capacidad de memorización y eficiencia.

### El Desafío de la Inferencia y la Invención del KV-Cache
Durante el entrenamiento, el Transformer procesa todos los tokens en paralelo ($O(1)$ pasos de GPU) gracias a la máscara causal.
Sin embargo, durante la **generación de texto en tiempo de inferencia**, los tokens deben producirse secuencialmente: $t_1 \to t_2 \to t_3$.
* Si en cada paso $t$ recalculamos las Keys y Values de todos los tokens pasados, el coste computacional crece como **$O(T^2)$**.
* **KV-Cache:** Al almacenar en memoria los tensores Key y Value calculados en pasos anteriores, generar cada nuevo token solo requiere computar una única Query $q_t$, reduciendo la inferencia a **$O(1)$ por token**.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### El Algoritmo Byte-Pair Encoding (BPE)
Un modelo de lenguaje no opera directamente con caracteres individuales (demasiado largo y lento) ni con palabras enteras (vocabulario infinito e inmanejable ante erratas o palabras raras):
1. Se inicia el vocabulario con los caracteres únicos del texto.
2. Se contabilizan todas las parejas de tokens adyacentes más frecuentes en el corpus.
3. El par más frecuente (ej. `'d'`, `'e'` $\to$ `'de'`) se fusiona creando un nuevo token en el vocabulario.
4. Se repite el proceso hasta alcanzar el tamaño de vocabulario deseado.

### La Anatomía del KV-Cache en Inferencia:
```
Paso 1: Prompt "El gato"
  Calcula: K_0, K_1 y V_0, V_1
  Almacena en KV-Cache: [K_0, K_1], [V_0, V_1]
  Genera token: "duerme"

Paso 2: Generar siguiente token
  Entrada: Solo el nuevo token "duerme" (pos=2)
  Calcula: Solo q_2, k_2, v_2 (¡no se reprocesan "El" ni "gato"!)
  Concatena al Cache: K_total = [K_cache, k_2], V_total = [V_cache, v_2]
  Atención: Softmax(q_2 @ K_total.T / sqrt(d)) @ V_total
  Genera token: "tranquilo"
```

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Construyamos primero un tokenizador BPE funcional y luego el modelo completo `NanoLLaMA` con soporte de KV-Cache.

In [ ]:
class SimpleBPETokenizer:
    """
    Tokenizador BPE (Byte-Pair Encoding) from scratch.
    """
    def __init__(self):
        self.merges: Dict[Tuple[int, int], int] = {}
        self.vocab: Dict[int, str] = {}
        self.inverse_vocab: Dict[str, int] = {}

    def train(self, text: str, vocab_size: int):
        # 1. Iniciar vocabulario con caracteres únicos
        chars = sorted(list(set(text)))
        self.vocab = {i: c for i, c in enumerate(chars)}
        self.inverse_vocab = {c: i for i, c in enumerate(chars)}
        
        # Representar el texto como lista de IDs de tokens iniciales
        ids = [self.inverse_vocab[c] for c in text]
        num_merges = vocab_size - len(chars)
        
        for i in range(num_merges):
            # Contar parejas adyacentes
            stats = {}
            for pair in zip(ids, ids[1:]):
                stats[pair] = stats.get(pair, 0) + 1
            if not stats:
                break
            # Encontrar el par más frecuente
            best_pair = max(stats, key=stats.get)
            new_id = len(self.vocab)
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            
            # Reemplazar el par en la lista de IDs
            new_ids = []
            idx = 0
            while idx < len(ids):
                if idx < len(ids) - 1 and (ids[idx], ids[idx+1]) == best_pair:
                    new_ids.append(new_id)
                    idx += 2
                else:
                    new_ids.append(ids[idx])
                    idx += 1
            ids = new_ids

    def encode(self, text: str) -> List[int]:
        tokens = [self.inverse_vocab.get(c, 0) for c in text]
        while len(tokens) >= 2:
            # Encontrar pares candidatos a fusionar
            stats = {pair: i for i, pair in enumerate(zip(tokens, tokens[1:]))}
            pair = min(stats.keys(), key=lambda p: self.merges.get(p, float('inf')))
            if pair not in self.merges:
                break
            idx_to_replace = self.merges[pair]
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                    new_tokens.append(idx_to_replace)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return tokens

    def decode(self, tokens: List[int]) -> str:
        return "".join([self.vocab.get(t, "") for t in tokens])

# Demostración del tokenizador
corpus = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivia un hidalgo..."
tokenizer = SimpleBPETokenizer()
tokenizer.train(corpus, vocab_size=50)
tokens_encoded = tokenizer.encode("de la Mancha")
print(f"Texto original: 'de la Mancha'")
print(f"Tokens BPE: {tokens_encoded}")
print(f"Texto decodificado: '{tokenizer.decode(tokens_encoded)}'")
print("✅ Tokenizador BPE from-scratch verificado")

### Construcción del Modelo NanoLLaMA con Pre-RMSNorm, RoPE y SwiGLU

Implementamos los componentes fundamentales de la arquitectura LLaMA:

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    """
    Feed-Forward con activación SwiGLU (Shazeer, 2020) estándar en LLaMA.
    """
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_up = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU(x) = (SiLU(x @ W_gate) * (x @ W_up)) @ W_down
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class CausalSelfAttentionWithKVCache(nn.Module):
    """
    Atención Causal con RoPE y soporte de aceleración mediante KV-Cache.
    """
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 512):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        B, T, D = x.shape
        H = self.n_heads
        d_k = self.head_dim
        
        q = self.q_proj(x).view(B, T, H, d_k).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, d_k).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, d_k).transpose(1, 2)
        
        # Si existe KV-Cache previo, concatenar las claves y valores históricos
        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k], dim=2)
            v = torch.cat([v_prev, v], dim=2)
        new_kv_cache = (k, v)
        
        # Atención por producto escalar escalada (usando F.scaled_dot_product_attention)
        # Si estamos en modo autorregresivo generando 1 token a la vez (T=1 con cache),
        # la máscara causal no es necesaria porque q solo atiende a posiciones pasadas.
        is_causal = (T > 1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out), new_kv_cache

print("✅ Módulos RMSNorm, SwiGLU y CausalSelfAttentionWithKVCache listos")

### Ensamblado de `NanoLLaMA` Completo
Conectamos el embedding de tokens, los bloques decodificadores con conexiones residuales y el cabezal de predicción de lenguaje (*LM Head*):

In [ ]:
class LLaMADecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttentionWithKVCache(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        self.mlp = SwiGLUMLP(d_model, hidden_dim=int(d_model * 2.67))

    def forward(self, x: torch.Tensor, kv_cache=None):
        # Pre-Norm residual para Atención
        attn_out, new_cache = self.attn(self.norm1(x), kv_cache=kv_cache)
        x = x + attn_out
        # Pre-Norm residual para SwiGLU
        x = x + self.mlp(self.norm2(x))
        return x, new_cache


class NanoLLaMA(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 64, n_heads: int = 4, n_layers: int = 3):
        super().__init__()
        self.vocab_size = vocab_size
        self.tok_embeddings = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([LLaMADecoderBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm_final = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # Compartición de pesos (Weight Tying) para optimizar memoria
        self.lm_head.weight = self.tok_embeddings.weight

    def forward(self, idx: torch.Tensor, kv_caches=None):
        B, T = idx.shape
        x = self.tok_embeddings(idx)
        
        new_caches = []
        for i, layer in enumerate(self.layers):
            cache_i = kv_caches[i] if kv_caches is not None else None
            x, new_c = layer(x, kv_cache=cache_i)
            new_caches.append(new_c)
            
        x = self.norm_final(x)
        logits = self.lm_head(x)
        return logits, new_caches

    @torch.no_grad()
    def generate(self, prompt_ids: List[int], max_new_tokens: int = 20, use_cache: bool = True) -> List[int]:
        self.eval()
        curr_tokens = list(prompt_ids)
        kv_caches = None
        
        if use_cache:
            # Procesar el prompt inicial para llenar el caché
            x_input = torch.tensor([curr_tokens], dtype=torch.long)
            logits, kv_caches = self.forward(x_input, kv_caches=None)
            next_token = torch.argmax(logits[:, -1, :], dim=-1).item()
            curr_tokens.append(next_token)
            
            # Generación acelerada 1 token a la vez reutilizando el KV-Cache
            for _ in range(max_new_tokens - 1):
                x_next = torch.tensor([[next_token]], dtype=torch.long)
                logits, kv_caches = self.forward(x_next, kv_caches=kv_caches)
                next_token = torch.argmax(logits[:, -1, :], dim=-1).item()
                curr_tokens.append(next_token)
        else:
            # Generación ingenua sin caché: recomputa toda la secuencia en cada paso O(T^2)
            for _ in range(max_new_tokens):
                x_input = torch.tensor([curr_tokens], dtype=torch.long)
                logits, _ = self.forward(x_input, kv_caches=None)
                next_token = torch.argmax(logits[:, -1, :], dim=-1).item()
                curr_tokens.append(next_token)
                
        return curr_tokens

print("✅ NanoLLaMA ensamblado exitosamente con generador autorregresivo y KV-Cache")

---

## 4. ⚡ Transición a PyTorch Moderno y Demostración

Entrenemos nuestro modelo durante unos pasos sobre una frase de entrenamiento para verificar que aprende a predecir los tokens correctamente:

In [ ]:
# Crear dataset de prueba
frase_entrenamiento = "El conocimiento es poder y el poder transforma el mundo."
vocabulario_chars = sorted(list(set(frase_entrenamiento)))
char_to_id = {c: i for i, c in enumerate(vocabulario_chars)}
id_to_char = {i: c for i, c in enumerate(vocabulario_chars)}
vocab_size = len(vocabulario_chars)

indices_texto = torch.tensor([char_to_id[c] for c in frase_entrenamiento], dtype=torch.long).unsqueeze(0)
x_train = indices_texto[:, :-1]
y_train = indices_texto[:, 1:]

modelo = NanoLLaMA(vocab_size=vocab_size, d_model=32, n_heads=2, n_layers=2)
optimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-2)

print("Entrenando NanoLLaMA en 60 pasos...")
modelo.train()
for paso in range(60):
    optimizador.zero_grad()
    logits, _ = modelo(x_train)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y_train.view(-1))
    loss.backward()
    optimizador.step()
    if (paso + 1) % 20 == 0:
        print(f"Paso {paso+1}/60 | Pérdida: {loss.item():.4f}")

# Generar texto con KV-Cache a partir de un prompt
prompt = "El cono"
prompt_ids = [char_to_id[c] for c in prompt]
generated_ids = modelo.generate(prompt_ids, max_new_tokens=25, use_cache=True)
texto_generado = "".join([id_to_char.get(i, "?") for i in generated_ids])
print(f"\nPrompt: '{prompt}'")
print(f"Texto Generado con KV-Cache: '{texto_generado}'")

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Benchmark de Rendimiento: Generación Sin Caché vs Con KV-Cache
Midamos experimentalmente la aceleración de inferencia que proporciona el KV-Cache al generar una secuencia larga:

In [ ]:
prompt_bench = [0, 1, 2]
n_tokens_a_generar = 60

# 1. Con KV-Cache
t0 = time.perf_counter()
_ = modelo.generate(prompt_bench, max_new_tokens=n_tokens_a_generar, use_cache=True)
tiempo_con_cache = (time.perf_counter() - t0) * 1000

# 2. Sin KV-Cache (Recomputando todo el contexto O(T^2))
t0 = time.perf_counter()
_ = modelo.generate(prompt_bench, max_new_tokens=n_tokens_a_generar, use_cache=False)
tiempo_sin_cache = (time.perf_counter() - t0) * 1000

print(f"Tiempo con KV-Cache: {tiempo_con_cache:.2f} ms")
print(f"Tiempo sin KV-Cache:  {tiempo_sin_cache:.2f} ms")
print(f"🚀 Aceleración: {tiempo_sin_cache / tiempo_con_cache:.2f}x más rápido con KV-Cache.")

### Reto 2 (Para resolver): Implementar Muestreo Nucleus (Top-$p$ Sampling)
En lugar de tomar siempre el token con mayor probabilidad (`argmax`), el muestreo con **Top-$p$** calcula las probabilidades acumuladas y filtra solo el subconjunto más probable hasta alcanzar una masa acumulada $p$ (ej. $p=0.9$):

Implementa la función `sample_top_p(logits, p=0.9, temperature=1.0)`:

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def sample_top_p(logits: torch.Tensor, p: float = 0.9, temperature: float = 1.0) -> int:
    """
    Aplica temperatura y muestreo Nucleus (Top-p) sobre un vector de logits de forma (vocab_size,).
    Devuelve el ID del token seleccionado.
    """
    # Tu implementación aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Radford, A., et al. (2019):** *"Language Models are Unsupervised Multitask Learners"* (GPT-2). OpenAI Technical Report. [OpenAI Link](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
   * *¿Qué leer?* La justificación del modelado de lenguaje autorregresivo con decodificadores puros.
2. **Touvron, H., et al. (2023):** *"LLaMA: Open and Efficient Foundation Language Models"*, Meta AI. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)
   * *¿Qué leer?* Sección 2 ("Approach"): la combinación de RMSNorm, RoPE y SwiGLU que definió el estándar contemporáneo.
3. **Shazeer, N. (2020):** *"GLU Variants Improve Transformer"*, arXiv:2002.05202. [arXiv Link](https://arxiv.org/abs/2002.05202)
   * *¿Qué leer?* La derivación y comparación empírica de SwiGLU frente a activaciones ReLU y GELU tradicionales.
4. **Pope, R., et al. (2023):** *"Efficiently Scaling Transformer Inference"*, MLSys 2023. [arXiv:2211.05102](https://arxiv.org/abs/2211.05102)
   * *¿Qué leer?* El análisis exhaustivo de cómo la memoria del KV-Cache domina la latencia y los costes de inferencia en clusters de GPUs.

### 🔗 Proyectos de Referencia
* **Andrej Karpathy:** [nanoGPT](https://github.com/karpathy/nanoGPT) - El repositorio más limpio y pedagógico para entrenar LLMs en PyTorch.